<a href="https://colab.research.google.com/github/smshozab/AG-AquaSense/blob/main/AAIn6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview

This notebook implements the **Retrieval-Augmented Generation (RAG) system** with continual updates.

### What this notebook does
- Builds synthetic knowledge base:
  - QA data
  - literature snippets
  - case records
- Splits documents into chunks
- Implements retrieval:
  - BM25 (keyword-based)
  - FAISS (semantic embeddings)
- Combines results using RRF (Reciprocal Rank Fusion)
- Simulates monthly updates to knowledge base

### Key outputs
- Retrieved knowledge chunks for queries
- ROUGE-L score (retrieval quality)
- Update cycle evaluation results

### Usage
- Provides knowledge context for Notebook 7

### Note
- Uses synthetic knowledge base (can be replaced with real data)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install faiss-cpu rank_bm25 sentence-transformers rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=97880b332262d3d1a21fc0333b09b48991eca5cb759a11bfb558fc49e138ae8b
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
import numpy as np
import pandas as pd
import random
from pathlib import Path

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss

from rouge_score import rouge_scorer

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/aquasense")
OUTPUT_DIR = BASE_DIR / "notebook6_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
knowledge_data = [
    # QA style
    {"type": "qa", "disease": "Bacterial Gill Disease",
     "content": "Fish gasping at surface usually indicates low dissolved oxygen or gill infection. Increase aeration immediately."},

    {"type": "qa", "disease": "Aeromoniasis",
     "content": "Ulcers and red lesions in fish are commonly caused by Aeromonas infection. Improve water quality and isolate infected fish."},

    {"type": "qa", "disease": "Saprolegniasis",
     "content": "Cotton-like white growth on fish skin indicates fungal infection such as saprolegniasis."},

    {"type": "qa", "disease": "Parasitic Diseases",
     "content": "Fish rubbing against tank surfaces suggests parasitic irritation."},

    {"type": "qa", "disease": "White Tail Disease",
     "content": "White discoloration of the tail muscle is a key symptom of white tail disease."},

    {"type": "qa", "disease": "Healthy Fish",
     "content": "Healthy fish swim actively, eat regularly, and show no visible lesions."},

    # Literature style
    {"type": "literature", "disease": "Bacterial Gill Disease",
     "content": "Bacterial gill disease is associated with hypoxia, increased ammonia, and poor water circulation."},

    {"type": "literature", "disease": "Aeromoniasis",
     "content": "Aeromoniasis is caused by Aeromonas bacteria and often occurs in warm water with high organic load."},

    {"type": "literature", "disease": "Saprolegniasis",
     "content": "Fungal infections thrive in stressed fish and cooler water conditions."},

    # Case records
    {"type": "case", "disease": "Bacterial Gill Disease",
     "content": "Tank A showed low DO levels and fish gasping. Lab confirmed gill disease."},

    {"type": "case", "disease": "Aeromoniasis",
     "content": "Tank B had high ammonia and fish with ulcers. Confirmed Aeromoniasis."},

    {"type": "case", "disease": "Parasitic Diseases",
     "content": "Fish exhibited flashing behavior and skin irritation. Parasites detected."},
]

kb_df = pd.DataFrame(knowledge_data)
display(kb_df)

,type,disease,content
0,qa,Bacterial Gill Disease,Fish gasping at surface usually indicates low ...
1,qa,Aeromoniasis,Ulcers and red lesions in fish are commonly ca...
2,qa,Saprolegniasis,Cotton-like white growth on fish skin indicate...
3,qa,Parasitic Diseases,Fish rubbing against tank surfaces suggests pa...
4,qa,White Tail Disease,White discoloration of the tail muscle is a ke...
5,qa,Healthy Fish,"Healthy fish swim actively, eat regularly, and..."
6,literature,Bacterial Gill Disease,Bacterial gill disease is associated with hypo...
7,literature,Aeromoniasis,Aeromoniasis is caused by Aeromonas bacteria a...
8,literature,Saprolegniasis,Fungal infections thrive in stressed fish and ...
9,case,Bacterial Gill Disease,Tank A showed low DO levels and fish gasping. ...


In [ ]:
def chunk_text(text, chunk_size=40):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

chunks = []

for _, row in kb_df.iterrows():
    text_chunks = chunk_text(row["content"])
    for c in text_chunks:
        chunks.append({
            "text": c,
            "disease": row["disease"],
            "type": row["type"]
        })

chunk_df = pd.DataFrame(chunks)
display(chunk_df.head())

,text,disease,type
0,Fish gasping at surface usually indicates low ...,Bacterial Gill Disease,qa
1,Ulcers and red lesions in fish are commonly ca...,Aeromoniasis,qa
2,Cotton-like white growth on fish skin indicate...,Saprolegniasis,qa
3,Fish rubbing against tank surfaces suggests pa...,Parasitic Diseases,qa
4,White discoloration of the tail muscle is a ke...,White Tail Disease,qa


In [ ]:
tokenized_corpus = [doc.split() for doc in chunk_df["text"]]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
embeddings = model.encode(chunk_df["text"].tolist(), show_progress_bar=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
def bm25_search(query, top_k=5):
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return top_idx, scores[top_idx]

In [ ]:
def faiss_search(query, top_k=5):
    q_emb = model.encode([query])
    distances, indices = index.search(q_emb, top_k)
    return indices[0], distances[0]

In [ ]:
def rrf_merge(bm25_idx, faiss_idx, k=60):
    scores = {}

    for rank, idx in enumerate(bm25_idx):
        scores[idx] = scores.get(idx, 0) + 1/(k + rank)

    for rank, idx in enumerate(faiss_idx):
        scores[idx] = scores.get(idx, 0) + 1/(k + rank)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in ranked]

In [ ]:
query = "fish gasping and breathing fast"

bm_idx, _ = bm25_search(query)
fa_idx, _ = faiss_search(query)

final_idx = rrf_merge(bm_idx, fa_idx)

results = chunk_df.iloc[final_idx[:5]]
display(results)

,text,disease,type
0,Fish gasping at surface usually indicates low ...,Bacterial Gill Disease,qa
5,"Healthy fish swim actively, eat regularly, and...",Healthy Fish,qa
8,Fungal infections thrive in stressed fish and ...,Saprolegniasis,literature
1,Ulcers and red lesions in fish are commonly ca...,Aeromoniasis,qa
9,Tank A showed low DO levels and fish gasping. ...,Bacterial Gill Disease,case


In [ ]:
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def evaluate_rouge(query, reference):
    bm_idx, _ = bm25_search(query)
    fa_idx, _ = faiss_search(query)
    final_idx = rrf_merge(bm_idx, fa_idx)

    retrieved_text = " ".join(chunk_df.iloc[final_idx[:3]]["text"])

    score = scorer.score(reference, retrieved_text)
    return score['rougeL'].fmeasure

In [ ]:
query = "fish rubbing and irritated skin"
reference = "Parasitic infections cause irritation and abnormal movement"

rouge_score = evaluate_rouge(query, reference)
print("ROUGE-L:", rouge_score)

ROUGE-L: 0.16216216216216217


In [ ]:
new_data = [
    {"type": "case", "disease": "White Tail Disease",
     "content": "New case: fish tail turned white and weak swimming observed."}
]

new_df = pd.DataFrame(new_data)

kb_df = pd.concat([kb_df, new_df], ignore_index=True)

In [ ]:
# Re-chunk
chunks = []
for _, row in kb_df.iterrows():
    for c in chunk_text(row["content"]):
        chunks.append({"text": c, "disease": row["disease"]})

chunk_df = pd.DataFrame(chunks)

# Rebuild BM25
tokenized_corpus = [doc.split() for doc in chunk_df["text"]]
bm25 = BM25Okapi(tokenized_corpus)

# Rebuild FAISS
embeddings = model.encode(chunk_df["text"].tolist())
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("Knowledge base updated.")

Knowledge base updated.


In [ ]:
results = []

for month in range(1, 4):
    rouge = evaluate_rouge(
        "fish breathing fast",
        "low oxygen causes gill disease"
    )

    results.append({
        "month": month,
        "rougeL": rouge
    })

results_df = pd.DataFrame(results)
display(results_df)

,month,rougeL
0,1,0.146341
1,2,0.146341
2,3,0.146341
